# Cross-validation by activation: every flood scored once

Notebooks [01](01_flood_segmentation_s1.ipynb) and [02](02_change_detection_s1.ipynb) scored their detectors on one fixed test split of four areas. Its independent part (EMS maps drawn on other satellites) holds only about 2 km² of mapped flood, while the large independent maps (EMSR664, 109 km²) sit in training. The scores on independent labels (CSI 0.03–0.07) rest on very little.

This notebook scores the same four detectors of notebook 02 with a **leave-one-activation-out cross-validation**: each activation is the test once, and the detectors are tuned and trained on the others only. Adding the contingency tables of all folds gives a score where **every mapped pixel counts once**, from a model that never saw its flood.

It sends nothing and spends **0 PU**: it reuses the pairs and references of notebook 02 from the data directory (run notebook 02 first, or download the `ems-data` artifact, see the [README](README.md)).

In [ ]:
import csv
import os
from pathlib import Path

import numpy as np

ACTIVATIONS = [
    "EMSR659",
    "EMSR664",
    "EMSR705",
    "EMSR762",
    "EMSR795",
    "EMSR800",
    "EMSR843",
    "EMSR855",
]
RESOLUTION_M = 40
CHIP = 128
SEED = 0

DATA_DIR = Path(os.environ.get("COPERNICUS_EMS_DATA_DIR", "~/.cache/copernicus-ems")).expanduser()
DATASET = DATA_DIR / "datasets" / f"italy-floods-s1-change-{RESOLUTION_M}m"
FIGURE = Path(os.environ.get("NOTEBOOK_FIGURE", "../docs/images/notebook-cv-s1.png"))
print("data:", DATA_DIR, "| dataset:", DATASET)

## 1. The five-band dataset of notebook 02

The dataset is rebuilt from the pairs in the data directory, locally and for free. Its train/val/test column is ignored: the folds are made by activation below.

In [ ]:
from copernicus_ems.dataset import build_dataset, find_pairs

pairs = find_pairs(DATA_DIR, ACTIVATIONS)
assert any(p.reference is not None for p in pairs), (
    "no pair with a reference: run notebook 02 first"
)
summary = build_dataset(pairs, DATASET, size=CHIP, min_observed=0.5, seed=SEED, reference=True)
rows = list(csv.DictReader((DATASET / "index.csv").open()))
made_from = {
    (p.activation, p.product): sorted({i["sensor"] for i in p.meta.get("made_from", [])})
    for p in pairs
}
for r in rows:
    r["made_from"] = ", ".join(made_from.get((r["activation"], r["product"]), []))
    r["same_image"] = "Sentinel-1" in r["made_from"]
FOLDS = sorted({r["activation"] for r in rows})
print(
    f"{summary.pairs} pairs -> {summary.chips} chips of {CHIP} px; {len(FOLDS)} folds: {', '.join(FOLDS)}"
)

## 2. Features

The eleven features of notebook 02: VV and VH in dB, their ratio, local mean and spread of VV, the pre-event VV and VH, and the changes ΔVV, ΔVH, raw and smoothed over 5×5 pixels.

In [ ]:
import rasterio
from scipy.ndimage import uniform_filter

from copernicus_ems.ml import FLOODED, IGNORE, LabelRaster

SINGLE = slice(0, 5)
NAMES = [
    "VV",
    "VH",
    "VV-VH",
    "VV mean",
    "VV spread",
    "VV pre",
    "VH pre",
    "dVV",
    "dVH",
    "dVV mean",
    "dVH mean",
]


def db(x):
    return 10 * np.log10(np.maximum(x, 1e-6))


def features(path):
    with rasterio.open(path) as src:
        vv, vh, vv_pre, vh_pre, _mask = src.read()
    vv_db, vh_db, vvp_db, vhp_db = db(vv), db(vh), db(vv_pre), db(vh_pre)
    mean = uniform_filter(vv_db, 5)
    spread = np.sqrt(np.maximum(uniform_filter(vv_db**2, 5) - mean**2, 0))
    dvv, dvh = vv_db - vvp_db, vh_db - vhp_db
    return np.stack(
        [
            vv_db,
            vh_db,
            vv_db - vh_db,
            mean,
            spread,
            vvp_db,
            vhp_db,
            dvv,
            dvh,
            uniform_filter(dvv, 5),
            uniform_filter(dvh, 5),
        ],
        axis=-1,
    ).astype(np.float32)


X = np.stack([features(DATASET / r["image"]) for r in rows])
LABELS = np.stack([LabelRaster.from_geotiff(DATASET / r["label"]).labels for r in rows])
Y = [LabelRaster.from_geotiff(DATASET / r["label"]) for r in rows]
km2 = (RESOLUTION_M / 1000) ** 2
print(
    f"{'activation':10} {'chips':>5} {'flooded km²':>12} {'independent km²':>16} {'same image km²':>15}"
)
for a in FOLDS:
    idx = [i for i, r in enumerate(rows) if r["activation"] == a]
    wet = [int((LABELS[i] == FLOODED).sum()) for i in idx]
    ind = sum(w for i, w in zip(idx, wet, strict=True) if not rows[i]["same_image"])
    print(
        f"{a:10} {len(idx):5d} {sum(wet) * km2:12.1f} {ind * km2:16.1f} {(sum(wet) - ind) * km2:15.1f}"
    )

## 3. One fold: tune and train on the other activations

Inside a fold the test activation is never touched:

* the VV threshold and the two change thresholds are chosen for the best CSI on the training chips;
* each classifier is fitted on a sample of training pixels (one flooded to three dry), and its probability cut is chosen on *other* training pixels, not in that sample.

These are the detectors of notebook 02; only the split changes.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

from copernicus_ems.ml import Scores, evaluate


def score(chips, predict):
    return sum((evaluate(predict(X[i]), Y[i]) for i in chips), Scores.zero())


def best(candidates, chips, make):
    return max(candidates, key=lambda c: score(chips, make(c)).csi or 0)


def fold(test_activation):
    train = [i for i, r in enumerate(rows) if r["activation"] != test_activation]
    t = best(np.arange(-30, -8, 0.5), train, lambda t: lambda f: f[..., 0] < t)
    t1, t2 = best(
        [(a, b) for a in np.arange(-24, 0.5, 1.0) for b in np.arange(-8, 0.1, 0.5)],
        train,
        lambda c: lambda f: (f[..., 0] < c[0]) & (f[..., 9] < c[1]),
    )
    keep = LABELS[train] != IGNORE
    x, y = X[train][keep], (LABELS[train][keep] == FLOODED).astype(np.uint8)
    rng = np.random.default_rng(SEED)
    wet, dry = rng.permutation(np.flatnonzero(y == 1)), rng.permutation(np.flatnonzero(y == 0))
    n = min(len(wet) // 2, 100_000)
    fit_px = np.concatenate([wet[:n], dry[: 3 * n]])
    cut_px = np.concatenate([wet[n : 2 * n], dry[3 * n : 6 * n]])

    def classifier(columns):
        model = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=SEED)
        model.fit(x[fit_px][:, columns], y[fit_px])
        p = model.predict_proba(x[cut_px][:, columns])[:, 1]
        truth = y[cut_px] == 1

        def csi(c):
            hits = int((truth & (p > c)).sum())
            return hits / max(int(truth.sum()) + int((~truth & (p > c)).sum()), 1)

        cut = max(np.arange(0.1, 0.95, 0.05), key=csi)

        def predict(f):
            g = f[..., columns]
            return model.predict_proba(g.reshape(-1, g.shape[-1]))[:, 1].reshape(f.shape[:-1]) > cut

        return predict

    return {
        "VV threshold": lambda f: f[..., 0] < t,
        "change threshold": lambda f: (f[..., 0] < t1) & (f[..., 9] < t2),
        "classifier 1 img": classifier(SINGLE),
        "classifier change": classifier(slice(0, len(NAMES))),
    }, {"t": float(t), "t1": float(t1), "t2": float(t2)}

## 4. Run the folds

For every fold the test chips are scored apart for independent labels (EMS maps drawn on another satellite) and same-image labels (drawn on this Sentinel-1 pass), inside the observation masks only.

In [ ]:
DETECTORS = ["VV threshold", "change threshold", "classifier 1 img", "classifier change"]
SUBSETS = {"independent": lambda r: not r["same_image"], "same image": lambda r: r["same_image"]}
per_fold = {}
for a in FOLDS:
    detectors, params = fold(a)
    test = [i for i, r in enumerate(rows) if r["activation"] == a]
    for name, predict in detectors.items():
        for subset, keep in SUBSETS.items():
            per_fold[a, name, subset] = score([i for i in test if keep(rows[i])], predict)
    print(
        f"{a}: VV < {params['t']:.1f} dB | change: VV < {params['t1']:.0f} dB and dVV < {params['t2']:.1f} dB"
    )

## 5. Scores, every flood counted once

The contingency tables of the folds are added, then CSI, POD, FAR and bias are computed from the sums: large floods weigh more, as they do in a map.

In [ ]:
pooled = {
    (name, subset): sum((per_fold[a, name, subset] for a in FOLDS), Scores.zero())
    for name in DETECTORS
    for subset in SUBSETS
}
print(
    f"{'':18} {'labels':12} {'CSI':>6} {'POD':>6} {'FAR':>6} {'F1':>6} {'bias':>6} {'flooded km²':>12}"
)
for subset in SUBSETS:
    for name in DETECTORS:
        s = pooled[name, subset]
        mapped = (s.hits + s.misses) * km2
        print(
            f"{name:18} {subset:12} {s.csi:6.3f} {s.pod:6.3f} {s.far:6.3f} {s.f1:6.3f} {s.bias:6.2f} {mapped:12.1f}"
        )

Per activation, CSI on its independent and same-image labels (`-` where it has none):

In [ ]:
def cell(s):
    return "     -" if s.hits + s.misses == 0 else f"{s.csi:6.3f}"


print(
    f"{'activation':10} {'labels':12}"
    + "".join(f"{n:>19}" for n in DETECTORS)
    + f"{'flooded km²':>13}"
)
for a in FOLDS:
    for subset in SUBSETS:
        first = per_fold[a, DETECTORS[0], subset]
        if first.hits + first.misses == 0:
            continue
        line = "".join(f"{cell(per_fold[a, n, subset]):>19}" for n in DETECTORS)
        print(f"{a:10} {subset:12}{line}{(first.hits + first.misses) * km2:13.1f}")

## 6. Figure

Pooled CSI of each detector, on independent and same-image labels.

In [ ]:
import matplotlib.pyplot as plt

COLORS = {"independent": "#2a78d6", "same image": "#eb6834"}
fig, ax = plt.subplots(figsize=(8, 3.6))
height = 0.36
for k, subset in enumerate(SUBSETS):
    y = np.arange(len(DETECTORS)) + (k - 0.5) * (height + 0.04)
    values = [pooled[n, subset].csi or 0 for n in DETECTORS]
    ax.barh(y, values, height=height, color=COLORS[subset], label=f"{subset} labels")
    for yy, v in zip(y, values, strict=True):
        ax.text(v + 0.005, yy, f"{v:.2f}", va="center", fontsize=9, color="#444")
ax.set_yticks(np.arange(len(DETECTORS)), DETECTORS)
ax.invert_yaxis()
ax.set_xlabel("CSI, leave-one-activation-out, all folds added")
ax.set_xlim(0, max(0.5, max(s.csi or 0 for s in pooled.values()) + 0.08))
ax.grid(axis="x", color="#e5e5e5")
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, loc="lower right")
fig.text(
    0.5,
    0.005,
    "Contains modified Copernicus Sentinel data and Copernicus EMS information, © European Union",
    ha="center",
    fontsize=7,
)
fig.tight_layout(rect=(0, 0.03, 1, 1))
FIGURE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE, dpi=110)
print("saved", FIGURE)

## Reading the scores

* **These are the numbers to quote.** Each flood is scored by detectors tuned and trained without it, and the independent labels now include the large EMSR664 maps instead of 2 km².
* **Folds are unequal.** EMSR664 holds most of the chips: when it is the test, the detectors learn from a few small floods only. The per-activation table shows how much the score moves from one flood to the next; a single average hides it.
* **Same-image labels still flatter.** They share their pixels with the input, so they measure agreement with the analyst's reading of that same image, not detection of an unmapped flood.

*Data: Copernicus EMS Rapid Mapping, © European Union; Copernicus Sentinel-1, processed by Sentinel Hub on the Copernicus Data Space Ecosystem.*